<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----
Уведомления

### Вариант задания  № 24


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Notification в C#, который будет представлять уведомления
пользователям. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма.
## Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:

public delegate bool NotificationFilter(Notification notification);

public delegate void NotificationSentHandler(object sender, NotificationEventArgs e);

public class NotificationEventArgs : EventArgs
{
    public string Status { get; }
    public DateTime Timestamp { get; }

    public NotificationEventArgs(string status)
    {
        Status = status;
        Timestamp = DateTime.Now;
    }
}

public class Notification
{
    public DateTime CreatedAt { get; protected set; }
    public int Priority { get; set; }
    public bool IsSent { get; protected set; }
    public List<string> Tags { get; set; }

    public event NotificationSentHandler NotificationSent;

    public int NotificationID { get; set; }
    public string MessageText { get; set; }
    public string Type { get; set; }

    public Notification(int id, string message, string type)
    {
        NotificationID = id;
        MessageText = message;
        Type = type;
        CreatedAt = DateTime.Now;
        Priority = 1;
        Tags = new List<string>();
        IsSent = false;
    }

    public virtual void SendNotification()
    {
        Console.WriteLine($"Отправка {Type} уведомления...");
        OnNotificationSent("Отправлено");
    }

    protected virtual void OnNotificationSent(string status)
    {
        IsSent = true;
        NotificationSent?.Invoke(this, new NotificationEventArgs(status));
    }

    public virtual void DisplayNotification()
    {
        Console.WriteLine($"Уведомление [{NotificationID}]: {MessageText}");
    }

    public virtual string GetNotificationDetails()
    {
        return $"ID: {NotificationID}, Тип: {Type}, Сообщение: {MessageText}, Приоритет: {Priority}";
    }

    public void AddTag(string tag)
    {
        Tags.Add(tag);
        Console.WriteLine($"Добавлен тег '{tag}' к уведомлению {NotificationID}");
    }

    public void UpdatePriority(int newPriority)
    {
        Priority = Math.Clamp(newPriority, 1, 5);
        Console.WriteLine($"Приоритет уведомления {NotificationID} изменен на {Priority}");
    }

    public TimeSpan GetAge()
    {
        return DateTime.Now - CreatedAt;
    }

    public virtual void MarkAsRead()
    {
        Console.WriteLine($"Уведомление {NotificationID} помечено как прочитанное");
    }
}

public class EmailNotification : Notification
{
    public string Subject { get; set; }
    public List<string> Attachments { get; set; }
    public bool IsHtml { get; set; }

    public string EmailAddress { get; set; }

    public EmailNotification(int id, string message, string email) : base(id, message, "Email")
    {
        EmailAddress = email;
        Subject = "Без темы";
        Attachments = new List<string>();
        IsHtml = false;
    }

    public override void SendNotification()
    {
        Console.WriteLine($"Отправка email на адрес {EmailAddress}");
        Console.WriteLine($"Тема: {Subject}");
        Console.WriteLine($"Текст: {MessageText}");
        if (Attachments.Any())
        {
            Console.WriteLine($"Приложения: {string.Join(", ", Attachments)}");
        }
        Console.WriteLine($"HTML-формат: {IsHtml}");
        
        OnNotificationSent("Email доставлен");
    }

    public override string GetNotificationDetails()
    {
        return base.GetNotificationDetails() + $", Email: {EmailAddress}, Тема: {Subject}";
    }

    public void AddAttachment(string filePath)
    {
        Attachments.Add(filePath);
        Console.WriteLine($"Добавлено вложение: {filePath}");
    }

    public void ValidateEmail()
    {
        bool isValid = EmailAddress.Contains("@") && EmailAddress.Contains(".");
        Console.WriteLine($"Валидация email {EmailAddress}: {(isValid ? "корректный" : "некорректный")}");
    }
}

public class SMSNotification : Notification
{
    public string SenderID { get; set; }
    public bool IsUnicode { get; set; }
    public int ValidityHours { get; set; }

    public string PhoneNumber { get; set; }

    public SMSNotification(int id, string message, string number) : base(id, message, "SMS")
    {
        PhoneNumber = number;
        SenderID = "DefaultSender";
        IsUnicode = false;
        ValidityHours = 24;
    }

    public override void SendNotification()
    {
        Console.WriteLine($"Отправка SMS на номер {PhoneNumber}");
        Console.WriteLine($"Отправитель: {SenderID}");
        Console.WriteLine($"Текст: {MessageText}");
        Console.WriteLine($"Юникод: {IsUnicode}, Срок действия: {ValidityHours}ч");
        
        OnNotificationSent("SMS отправлено");
    }

    public override string GetNotificationDetails()
    {
        return base.GetNotificationDetails() + $", Номер телефона: {PhoneNumber}, Отправитель: {SenderID}";
    }

    public void ValidatePhoneNumber()
    {
        bool isValid = PhoneNumber.StartsWith("+") && PhoneNumber.Length >= 10;
        Console.WriteLine($"Валидация номера {PhoneNumber}: {(isValid ? "корректный" : "некорректный")}");
    }

    public void ExtendValidity(int additionalHours)
    {
        ValidityHours += additionalHours;
        Console.WriteLine($"Срок действия SMS продлен до {ValidityHours} часов");
    }
}

public class PushNotification : Notification
{
    public string AppVersion { get; set; }
    public int BadgeCount { get; set; }
    public Dictionary<string, string> CustomData { get; set; }

    public string Platform { get; set; }

    public PushNotification(int id, string message, string platform) : base(id, message, "Push")
    {
        Platform = platform;
        AppVersion = "1.0";
        BadgeCount = 0;
        CustomData = new Dictionary<string, string>();
    }

    public override void DisplayNotification()
    {
        Console.WriteLine($"[{Platform}] Push-уведомление: {MessageText}");
        if (BadgeCount > 0)
        {
            Console.WriteLine($"Количество badge: {BadgeCount}");
        }
    }

    public override void SendNotification()
    {
        Console.WriteLine($"Отправка push-уведомления на платформу {Platform}");
        Console.WriteLine($"Версия приложения: {AppVersion}");
        Console.WriteLine($"Текст: {MessageText}");
        
        if (CustomData.Any())
        {
            Console.WriteLine("Дополнительные данные:");
            foreach (var data in CustomData)
            {
                Console.WriteLine($"  {data.Key}: {data.Value}");
            }
        }
        
        OnNotificationSent("Push доставлен");
    }

    public override string GetNotificationDetails()
    {
        return base.GetNotificationDetails() + $", Платформа: {Platform}, Версия: {AppVersion}";
    }

    public void AddCustomData(string key, string value)
    {
        CustomData[key] = value;
        Console.WriteLine($"Добавлены данные: {key} = {value}");
    }

    public void SetBadge(int count)
    {
        BadgeCount = count;
        Console.WriteLine($"Установлен badge count: {count}");
    }
}

public class NotificationManager
{
    private List<Notification> _notifications;
    public event Action<string> OnLog;

    public NotificationManager()
    {
        _notifications = new List<Notification>();
    }

    public void AddNotification(Notification notification)
    {
        _notifications.Add(notification);
        notification.NotificationSent += (sender, e) => 
        {
            OnLog?.Invoke($"Лог: {((Notification)sender).Type} уведомление отправлено со статусом: {e.Status} в {e.Timestamp}");
        };
        
        OnLog?.Invoke($"Добавлено новое уведомление: {notification.Type} (ID: {notification.NotificationID})");
    }

    public List<Notification> FilterNotifications(NotificationFilter filter)
    {
        return _notifications.Where(n => filter(n)).ToList();
    }

    public void ProcessHighPriorityNotifications()
    {
        var highPriority = _notifications.Where(n => n.Priority >= 4).ToList();
        OnLog?.Invoke($"Найдено уведомлений с высоким приоритетом: {highPriority.Count}");
        
        foreach (var notification in highPriority)
        {
            notification.SendNotification();
        }
    }

    public void DisplayStatistics()
    {
        var stats = _notifications
            .GroupBy(n => n.Type)
            .Select(g => new { Type = g.Key, Count = g.Count() });
        
        Console.WriteLine("\n=== Статистика уведомлений ===");
        foreach (var stat in stats)
        {
            Console.WriteLine($"{stat.Type}: {stat.Count}");
        }
    }
}


        Console.WriteLine("Демонстрация системы уведомлений:");
        Console.WriteLine("==================================");

        var manager = new NotificationManager();
        manager.OnLog += (message) => Console.WriteLine($"[Лог] {message}");

        var email = new EmailNotification(1, "Добро пожаловать в нашу систему!", "user@example.com")
        {
            Subject = "Приветствие",
            IsHtml = true
        };
        email.AddAttachment("welcome.pdf");
        email.UpdatePriority(3);

        var sms = new SMSNotification(2, "Ваш код подтверждения: 1234", "+79991234567")
        {
            SenderID = "MyApp",
            ValidityHours = 48
        };
        sms.AddTag("важно");
        sms.AddTag("подтверждение");

        var push = new PushNotification(3, "У вас новое сообщение", "Android")
        {
            AppVersion = "2.1.0",
            BadgeCount = 5
        };
        push.AddCustomData("message_id", "12345");
        push.AddCustomData("sender", "Иван Иванов");

        manager.AddNotification(email);
        manager.AddNotification(sms);
        manager.AddNotification(push);

        Console.WriteLine("\n--- Фильтрация уведомлений ---");
        NotificationFilter importantFilter = (n) => n.Tags.Contains("важно") || n.Priority >= 4;
        var importantNotifications = manager.FilterNotifications(importantFilter);
        
        Console.WriteLine($"Важных уведомлений: {importantNotifications.Count}");
        foreach (var notification in importantNotifications)
        {
            Console.WriteLine($"- {notification.GetNotificationDetails()}");
        }

        Console.WriteLine("\n--- Обработка уведомлений ---");
        var notifications = new List<Notification> { email, sms, push };
        
        foreach (var notification in notifications)
        {
            Console.WriteLine("\n--- Обработка уведомления ---");
            notification.DisplayNotification();
            notification.SendNotification();
            Console.WriteLine("Детали: " + notification.GetNotificationDetails());
            
            if (notification is EmailNotification emailNotification)
            {
                emailNotification.ValidateEmail();
            }
            else if (notification is SMSNotification smsNotification)
            {
                smsNotification.ValidatePhoneNumber();
            }
            else if (notification is PushNotification pushNotification)
            {
                pushNotification.MarkAsRead();
            }
        }

        Console.WriteLine("\n--- Обработка высокоприоритетных уведомлений ---");
        manager.ProcessHighPriorityNotifications();

        manager.DisplayStatistics();

        Console.WriteLine("\n==================================");
        Console.WriteLine("Обработка всех уведомлений завершена!");


Демонстрация системы уведомлений:
Добавлено вложение: welcome.pdf
Приоритет уведомления 1 изменен на 3
Добавлен тег 'важно' к уведомлению 2
Добавлен тег 'подтверждение' к уведомлению 2
Добавлены данные: message_id = 12345
Добавлены данные: sender = Иван Иванов
[Лог] Добавлено новое уведомление: Email (ID: 1)
[Лог] Добавлено новое уведомление: SMS (ID: 2)
[Лог] Добавлено новое уведомление: Push (ID: 3)

--- Фильтрация уведомлений ---
Важных уведомлений: 1
- ID: 2, Тип: SMS, Сообщение: Ваш код подтверждения: 1234, Приоритет: 1, Номер телефона: +79991234567, Отправитель: MyApp

--- Обработка уведомлений ---

--- Обработка уведомления ---
Уведомление [1]: Добро пожаловать в нашу систему!
Отправка email на адрес user@example.com
Тема: Приветствие
Текст: Добро пожаловать в нашу систему!
Приложения: welcome.pdf
HTML-формат: True
[Лог] Лог: Email уведомление отправлено со статусом: Email доставлен в 11/16/2025 11:38:47 PM
Детали: ID: 1, Тип: Email, Сообщение: Добро пожаловать в нашу систему!, 